# Chapter 9 Lab — Triton Model Repository + config.pbtxt + Mock Client

**Estimated time:** 25 minutes · **Read-and-run.** We build the repo layout in memory, parse a
`config.pbtxt`, and send a request to a *stub* server — no real Triton needed.

In [ ]:
model_repo = {
  'llama/' : ['config.pbtxt', '1/model.plan'],   # '1' = version dir, model.plan = TRT engine
}
for d, files in model_repo.items():
    print(d); [print('   ', f) for f in files]

In [ ]:
config_pbtxt = '''
name: "llama"
backend: "tensorrtllm"
max_batch_size: 8
input  [ { name: "text_input"  data_type: TYPE_STRING dims: [ -1 ] } ]
output [ { name: "text_output" data_type: TYPE_STRING dims: [ -1 ] } ]
'''
def pb_get(txt, key):
    for line in txt.splitlines():
        s=line.strip()
        if s.startswith(key+':'): return s.split(':',1)[1].strip().strip('"')
    return None
print('backend       =', pb_get(config_pbtxt,'backend'))
print('max_batch_size=', pb_get(config_pbtxt,'max_batch_size'))

In [ ]:
# Stub 'server' + client round-trip.
def triton_server(request):
    assert 'text_input' in request, 'missing required input tensor'
    return {'text_output': f"echo: {request['text_input']}"}
def client_infer(text):
    return triton_server({'text_input': text})
print(client_infer('Explain NVIDIA NIM in one line.'))

## Key takeaway
**NIM** accelerates standard deployments with prebuilt, optimized microservices; **Triton** gives
granular control (custom backends, ensembles, dynamic batching, multiple versions). The exam tests
recognizing which situation calls for which.